In [1]:
import re
from collections import Counter


In [2]:
!wget https://norvig.com/big.txt

--2026-06-11 16:54:59--  https://norvig.com/big.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6488666 (6.2M) [text/plain]
Saving to: ‘big.txt’

big.txt             100%[===================>]   6.19M  18.6MB/s    in 0.3s    

2026-06-11 16:55:00 (18.6 MB/s) - ‘big.txt’ saved [6488666/6488666]



In [3]:
def words(text):
    return re.findall(r'\w+', text.lower())

with open('big.txt') as f:
    WORDS = Counter(words(f.read()))

print("Total unique words:", len(WORDS))


Total unique words: 32198


In [4]:
TOTAL_WORDS = sum(WORDS.values())

def P(word):
    """Probability of `word`."""
    return WORDS[word] / TOTAL_WORDS


In [5]:
def edits1(word):
    letters = 'abcdefghijklmnopqrstuvwxyz'
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]


In [6]:
def edits1(word):
    letters = 'abcdefghijklmnopqrstuvwxyz'
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

    deletes = [L + R[1:] for L, R in splits if R]
    inserts = [L + c + R for L, R in splits for c in letters]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]

    return set(deletes + inserts + replaces + transposes)


In [7]:
def known(words):
    return set(w for w in words if w in WORDS)


In [8]:
def candidates(word):
    return (known([word]) or
            known(edits1(word)) or
            known(e2 for e1 in edits1(word) for e2 in edits1(e1)) or
            [word])


In [9]:
def correct(word):
    return max(candidates(word), key=P)


In [10]:
def correct_sentence(sentence):
    tokens = re.findall(r'\w+|\W+', sentence)

    corrected_tokens = []
    for token in tokens:
        if token.isalpha():
            corrected_tokens.append(correct(token.lower()))
        else:
            corrected_tokens.append(token)

    return ''.join(corrected_tokens)


In [11]:
def autocorrect_chatbot():
    print("🧠 Autocorrect Chatbot")
    print("Type 'exit' to quit.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == 'exit':
            print("Bot: Goodbye! 👋")
            break

        corrected = correct_sentence(user_input)
        print("Bot:", corrected)


In [12]:
autocorrect_chatbot()

🧠 Autocorrect Chatbot
Type 'exit' to quit.

You: sprig
Bot: sprig
You: sprang
Bot: sprang
You: acessories
Bot: acessories
You: spring
Bot: spring
You: wring
Bot: wring
You: corect
Bot: correct
You: meanig
Bot: meaning
You: exite
Bot: exit
You: exit
Bot: Goodbye! 👋
